# AFESM — per-cluster taxonomic composition

Builds **`afesm_cluster_composition.geq{MIN_TOTAL}[.noFlag3].tsv.gz`**: one row per AFESM cluster, with integer protein counts per taxonomic category and per eukaryotic supergroup.

Eukaryotes that cannot be placed in a supergroup are pooled in a single **`E_Unassigned`** column: domain-only lineages (`cellular organisms; Eukaryota`) together with `environmental samples`, `unclassified eukaryotes` and `Eukaryota incertae sedis` (the former `E_Other`).

Asgard proteins are split by **GTDB class** into `As_<class>` columns (`As_Lokiarchaeia`, `As_Thorarchaeia`, `As_Heimdallarchaeia`, `As_Njordarchaeia`, …), and **`As`** holds their sum, as `E_total` does for eukaryotes. GTDB-filed taxa use their class directly; NCBI-filed taxa (*Asgard group / Promethearchaeati; Candidatus <phylum>*) are mapped onto the GTDB class (Heimdall-, Hod-, Gerd-, Kariarchaeota → Heimdallarchaeia; Loki-/Promethe-, Hel-, Sigynarchaeota → Lokiarchaeia; Borrarchaeota → Sifarchaeia; others (Frey-, Frigg-, Gefion-, Idunn-, Wukong-) → their own `<stem>archaeia` column); Panguiarchaeales (filed under GTDB *Thermoproteota; JANJXX01*) → **Njordarchaeia**. `incertae sedis` ranks are skipped (the next rank down is used); taxa resolved only to phylum (`Asgardarchaeota`, *Asgard group archaeon*, *Asgard group incertae sedis*) go to `As_unresolved`. Cell 5b can regroup the columns without re-running the pass.

Classification uses `categorize_lineage()` / `EUK_GROUPS` from **cell 3 of `5_AsgardEuk_taxonomy.ipynb`**, with one change: `E_Other` is merged into `E_Unassigned`.

---

### ⚙️ Options (set in the *Setup* cell)

| Parameter | Meaning |
|---|---|
| `MIN_TOTAL` | minimum proteins per cluster (default 20) |
| `DROP_FLAG3` | drop the low-confidence clusters: whole MMseqs2 sequence clusters whose members all have `cluFlag == 3` (representative structure pLDDT < 60), and which therefore were never merged into a structural cluster; see below |
| `APPLY_ASGARD_FIX` | reclassify every UniProt member filed under the composite taxID 4137846 from its own UniProt (or UniSave) lineage: As / A / uA (step 2b). Default `True` |

**About `cluFlag` and `DROP_FLAG3`.** AFESM was built in two steps (AFESM paper, Methods). Proteins were first grouped into sequence clusters with MMseqs2 (`--min-seq-id 0.3 --cov-mode 1 -c 0.9`). Each sequence cluster's representative was set to its highest-pLDDT member among those longer than 70% of the longest one. The 84,463,443 sequence clusters whose representative had pLDDT < 60 (151,488,016 proteins) were excluded. The remaining 102,104,329 representatives were then clustered by structure with Foldseek into 72.6 M structural clusters.

The member table therefore contains two kinds of `repId`. Flag-1/2 rows belong to Foldseek **structural clusters**. Flag-3 rows belong to the excluded MMseqs2 **sequence clusters**, which were never merged into any structural cluster. A `repId` is therefore either entirely flag 3 or contains no flag-3 member at all; this table has 84,463,442 flag-3-only `repId`s (151,488,013 proteins) and 72,604,896 structural clusters, matching the paper. `DROP_FLAG3 = True` removes these whole sequence clusters and never trims a structural cluster. With `MIN_TOTAL = 20` it drops 460,379 sequence clusters (32.2 M proteins) of the 2,355,093 `repId`s with ≥20 members, keeping the 1,894,714 structural clusters (`cluFlag12 == 1`). The code skips flag-3 rows and then applies `MIN_TOTAL`; since flag-3-only `repId`s are left with 0 proteins, this is the same as dropping them outright.

`cluFlag12` is always written as a column, so the *cluster-level* filter used in the reference notebook (drop `repId`s with no flag-1/2 member) can be applied afterwards without re-running.

### ⏱️ Runtime

- 7.2 GB download (10–30 min)
- ~60–90 min streaming pass over 820M rows
- ~8 GB disk

**Shortcut:** the resulting table for the default settings (`MIN_TOTAL=20`, `DROP_FLAG3=True`, `APPLY_ASGARD_FIX=True`) is in the GitHub repository as `afesm_cluster_composition.geq20.noFlag3.slim.tsv.gz`. Run the notebook only to rebuild it or to use other settings.

### ✅ Expected output (asserted by the last cell)

Within the `cluFlag12 == 1` scope the per-category protein totals reproduce the **reference notebook** (`1_clusters_with_20counts_treemap_for_taxonomy_4137846fix.ipynb`, an independent implementation of the same counts) exactly, **including the `A` / `uA` split**: `As` = 442,684; `A` = 12,369,794; `uA` = 307,306; `B` = 448,066,625; `E` = 44,319,915; `U` = 46,887,887; `V` = 3,567,776. The `As` / `A` / `uA` figures only hold with `APPLY_ASGARD_FIX = True` (the reference notebook applies the same 4137846 reclassification). This notebook also sends archaeal taxa whose lineage stops at the domain (`environmental samples`, `unclassified Archaea`, `Archaea incertae sedis`) to `uA`; the reference notebook has no such rule, but those taxa contribute **zero** proteins in this scope, so the split is unchanged.


| `MIN_TOTAL` | `DROP_FLAG3` | clusters |
|---|---|---|
| 20 | `False` | 2,355,093 (of which 1,894,714 have `cluFlag12 == 1`) |
| 20 | `True`  | 1,894,714 |

> These agree because in AFESM **every** `cluFlag == 3` member sits in a cluster with no flag-1/2 member at all (verified: such clusters hold 32,213,031 proteins in the ≥20 scope, exactly the reference notebook's flag-3 tally). So removing flag-3 *members* and dropping flag-3-only *clusters* are the same operation here, and `DROP_FLAG3=True` reproduces the reference notebook's scope.

## Setup
Choose the filters, then run all cells (*Runtime → Run all*).

In [1]:
#@title Setup
MIN_TOTAL  = 20      #@param {type:"integer"}
DROP_FLAG3 = True   #@param {type:"boolean"}
APPLY_ASGARD_FIX = True   #@param {type:"boolean"}
# DROP_FLAG3=False -> counts every member; `total` == `total_all`
# DROP_FLAG3=True  -> cluFlag==3 members are excluded from all category counts and from
#                     `total`; MIN_TOTAL then applies to what remains. `total_all` and
#                     `n_flag3` still record what was there before the cut. Note MIN_TOTAL
#                     is applied to the post-removal count; in AFESM no cluster mixes flag-3
#                     with flag-1/2 members, so this makes no difference to the real data.

import os, subprocess
import pandas as pd

BASE    = "https://afesm.steineggerlab.workers.dev/v1/"
TAX_TGZ = "7-taxonomy.tar.gz"
MEMBERS = "3-repId_memId_cluFlag_taxId_biomeId.tsv.gz"
MEMBERS_BYTES = 7186445769
# UniProt annotation of every member on taxID 4137846 (notebook 1, cell 40)
ASGARD_FIX_GDRIVE_ID = "1yE9vNuXvDCmgMLA7r7cur-oblrosK5US"
N_CLUSTERS    = 157068338      # distinct repIds; the pass asserts it saw exactly this many
OUT = f"afesm_cluster_composition.geq{MIN_TOTAL}{'.noFlag3' if DROP_FLAG3 else ''}.tsv.gz"
os.makedirs("afesm", exist_ok=True)
print(f"MIN_TOTAL={MIN_TOTAL}  DROP_FLAG3={DROP_FLAG3}  APPLY_ASGARD_FIX={APPLY_ASGARD_FIX}\noutput -> {OUT}")

MIN_TOTAL=20  DROP_FLAG3=True  APPLY_ASGARD_FIX=True
output -> afesm_cluster_composition.geq20.noFlag3.tsv.gz


## 1 · Taxonomy dump
Downloads the AFESM NCBI-style taxonomy and builds a `taxID → lineage` lookup.

In [2]:
#@title 1. Taxonomy dump -> taxID: lineage
# curl, not urllib: the Cloudflare front end answers Python's default User-Agent with 403.
tgz = "afesm/tax.tar.gz"
if not (os.path.exists(tgz) and subprocess.run(["gzip", "-t", tgz]).returncode == 0):
    subprocess.run(["curl", "-sS", "--fail", "--location", "--retry", "5", "-o", tgz, BASE + TAX_TGZ],
                   check=True)
subprocess.run(["tar", "-xzf", tgz, "-C", "afesm"], check=True)
D = os.path.dirname(subprocess.run(["find", "afesm", "-name", "afesm_nodes.dmp"],
                                   capture_output=True, text=True).stdout.strip())

nodes, names, merged = {}, {}, {}
with open(f"{D}/afesm_nodes.dmp") as fh:
    for ln in fh:
        f = [x.strip() for x in ln.split("|")]; nodes[int(f[0])] = int(f[1])
with open(f"{D}/afesm_names.dmp") as fh:
    for ln in fh:
        f = [x.strip() for x in ln.split("|")]
        # The dump also carries blast synonyms ("prokaryotes", "eukaryotes"). Without this
        # filter the lineages are wrong and every taxon falls through to 'U'.
        if len(f) > 3 and f[3] == "scientific name":
            names[int(f[0])] = f[1]
with open(f"{D}/afesm_merged.dmp") as fh:
    for ln in fh:
        f = [x.strip() for x in ln.split("|")]; merged[int(f[0])] = int(f[1])

def lineage(t):
    """Root-to-tip '; '-joined lineage, excluding the root node."""
    t = merged.get(t, t); out = []; seen = set()
    while t in nodes and t not in seen and t != 1:
        seen.add(t); out.append(names.get(t, f"Unknown_{t}")); t = nodes[t]
    return "; ".join(reversed(out))

assert lineage(2157) == "cellular organisms; Archaea", "names parse looks wrong"
print(f"{len(nodes):,} nodes | {len(merged):,} merged")

2,717,341 nodes | 74,915 merged


## 2 · Lineage classification
`categorize_lineage()` + `EUK_GROUPS`, from `5_AsgardEuk_taxonomy.ipynb` (cell 3). Writes `afesm/taxmap.tsv`.

In [3]:
#@title 2. categorize_lineage + EUK_GROUPS  (5_AsgardEuk_taxonomy.ipynb cell 3)
import re
ASGARD_KEYS = ('asgard', 'promethearchaeati', 'panguiarchaeales')
# Panguiarchaeales: genome-reduced Asgard order within Njordarchaeia, MBE (2025)
# https://doi.org/10.1093/molbev/msaf201. GTDB files it under Thermoproteota, so without
# this key its ~31k proteins are not counted as Asgard.
UA_RANKS = {'environmental samples', 'unclassified archaea', 'archaea incertae sedis'}

EUK_GROUPS = [  # first match wins; exact rank names, case-insensitive
    ('metazoa', 'E_O_Metazoa'), ('fungi', 'E_O_Fungi'), ('opisthokonta', 'E_O_other'),
    ('viridiplantae', 'E_Archaeplastida'), ('rhodophyta', 'E_Archaeplastida'),
    ('glaucocystophyceae', 'E_Archaeplastida'), ('rhodelphea', 'E_Archaeplastida'),
    ('picozoa', 'E_Archaeplastida'), ('palpitomonas', 'E_Archaeplastida'),
    ('stramenopiles', 'E_Sar_Stramenopiles'), ('alveolata', 'E_Sar_Alveolata'),
    ('rhizaria', 'E_Sar_Rhizaria'), ('penardia', 'E_Sar_Rhizaria'), ('sar', 'E_Sar_unknown'),
    ('haptista', 'E_Haptista'), ('telonemia', 'E_Haptista'), ('telonemida', 'E_Haptista'),
    ('cryptophyceae', 'E_Cryptophyceae'), ('amoebozoa', 'E_Amoebozoa'),
    # excavates: Nature 640, 974-981 (2025) https://doi.org/10.1038/s41586-025-08709-5
    ('discoba', 'E_basal_Discoba'), ('metamonada', 'E_basal_Metamonada'),
    ('malawimonadida', 'E_basal_Malawimonadida'),
    ('hemimastigophora', 'E_Hemimastigophora'), ('apusozoa', 'E_Apusozoa'),
    ('breviatea', 'E_Breviatea'), ('provora', 'E_Provora'), ('ancyromonadida', 'E_Ancyromonadida'),
    ('crums', 'E_CRuMs'), ('collodictyonidae', 'E_CRuMs'), ('rigifilida', 'E_CRuMs'),
    ('mantamonas', 'E_CRuMs'), ('meteora', 'E_CRuMs'), ('microheliella', 'E_CRuMs'),
    ('dimorpha', 'E_CRuMs'),
]

def categorize_lineage(lin):
    parts = [p.strip() for p in str(lin).split(';')]
    if parts and parts[0] == 'cellular organisms':
        parts = parts[1:]
    top = parts[0] if parts else ''
    if top == 'Viruses':  return 'V'
    if top == 'Bacteria': return 'B'
    if top == 'Archaea':
        ll = str(lin).lower()
        if any(k in ll for k in ASGARD_KEYS): return 'As'
        if len(parts) == 1 or parts[1].lower() in UA_RANKS: return 'uA'
        return 'A'
    if top == 'Eukaryota':
        ranks = {p.lower() for p in parts[1:]}
        for tok, lab in EUK_GROUPS:
            if tok in ranks:
                return lab
        # no supergroup: domain-only, environmental samples, unclassified eukaryotes,
        # Eukaryota incertae sedis -> one pooled class (formerly E_Other + E_Unassigned)
        return 'E_Unassigned'
    return 'U'

# ---- Asgard class (GTDB classes, as used by the AFESM dump)
# The dump files most Asgard taxa under GTDB (Asgardarchaeota; <class>; ...), some under NCBI
# (Asgard group / Promethearchaeati; Candidatus <phylum>), and Panguiarchaeales under GTDB
# Thermoproteota; JANJXX01. Everything is reported at GTDB class level:
AS_SKIP = {"cellular organisms", "archaea", "promethearchaeati", "asgard group",
           "asgardarchaeota", "candidatus asgardarchaeota"}
# NCBI phylum stem -> GTDB class (stems not listed become <stem>archaeia)
NCBI2CLASS = {
    "Loki": "Lokiarchaeia", "Promethe": "Lokiarchaeia",   # Promethearchaeota = renamed Lokiarchaeota
    "Hel": "Lokiarchaeia",                                # Helarchaeales sit in Lokiarchaeia (GTDB)
    "Hod": "Heimdallarchaeia", "Gerd": "Heimdallarchaeia", "Kari": "Heimdallarchaeia",  # orders/families of Heimdallarchaeia
    "Sigyn": "Lokiarchaeia",   # the dump itself files Sigynarchaeales under Lokiarchaeia
    "Borr": "Sifarchaeia",     # the dump itself files Borrarchaeales under Sifarchaeia
    # Frey-, Frigg-, Gefion-, Idunn-, Wukongarchaeota are kept as their own <stem>archaeia column;
    # pool them in cell 5b (AS_GROUP) if you place them in a GTDB class.
}
PANGUI_CLASS = "Njordarchaeia"   # Panguiarchaeales placed within Njordarchaeia (MBE 2025)

def asgard_class(lin):
    """GTDB class of an Asgard lineage; 'unresolved' when it stops above class level."""
    parts = [x.strip() for x in str(lin).split(";")]
    if any(x.lower() == "panguiarchaeales" for x in parts):
        return PANGUI_CLASS
    for x in parts:
        q = x.lower()
        if q in AS_SKIP or "incertae sedis" in q:   # e.g. 'Asgard group incertae sedis': look one rank deeper
            continue
        if q.startswith(("unclassified", "environmental")) or " archaeon" in q:
            return "unresolved"
        name = re.sub(r"^candidatus\s+", "", x, flags=re.I)
        if name.endswith("archaeia") or "archae" not in name.lower():   # GTDB class, incl. placeholders (JAGGXT01)
            return name
        stem = name[:name.lower().find("archae")]                         # NCBI phylum
        return NCBI2CLASS.get(stem, stem + "archaeia")
    return "unresolved"

def as_label(lin):
    """Column-safe label: awk receives the column list space-separated, so no spaces allowed."""
    return "As_" + re.sub(r"\W+", "_", asgard_class(lin)).strip("_")

for lin, want in [
    ("cellular organisms; Archaea; Asgardarchaeota; Lokiarchaeia; Sigynarchaeales", "Lokiarchaeia"),
    ("cellular organisms; Archaea; Asgardarchaeota; JAGGXT01; JAGGXT01", "JAGGXT01"),
    ("cellular organisms; Archaea; Asgardarchaeota", "unresolved"),
    ("cellular organisms; Archaea; Asgard group; Candidatus Heimdallarchaeota; unclassified Candidatus Heimdallarchaeota", "Heimdallarchaeia"),
    ("cellular organisms; Archaea; Asgard group; Candidatus Gerdarchaeota", "Heimdallarchaeia"),
    ("cellular organisms; Archaea; Promethearchaeati; Promethearchaeota", "Lokiarchaeia"),
    ("cellular organisms; Archaea; Thermoproteota; JANJXX01; Panguiarchaeales; Panguiarchaeaceae", "Njordarchaeia"),
    ("cellular organisms; Archaea; Asgard group; Asgard group archaeon", "unresolved"),
    ("cellular organisms; Archaea; Asgard group; Asgard group incertae sedis", "unresolved"),
    ("cellular organisms; Archaea; Asgard group; Asgard group incertae sedis; Candidatus Sigynarchaeota", "Lokiarchaeia"),
    ("cellular organisms; Archaea; Asgard group; Candidatus Borrarchaeota", "Sifarchaeia"),
    ("cellular organisms; Archaea; Asgard group; Candidatus Wukongarchaeota", "Wukongarchaeia"),
]:
    assert asgard_class(lin) == want, (lin, asgard_class(lin))

MGYP4137846_TAXID = 41378460000000000   # MGYP members on 4137846 -> uA (no lineage below Archaea)
SPECIFIC_AS       = [4138202, 4140603]   # 4139494 deliberately NOT forced (Hadarchaeota)
RADIX             = [1110365, 1110366]   # unclassified Radix (Mollusca), absent from the dump

tax = pd.DataFrame({"taxID": list(nodes)})
tax["label"] = [categorize_lineage(lineage(t)) for t in tax.taxID]
tax.loc[tax.taxID.isin(SPECIFIC_AS), "label"] = "As"
# split Asgard by class: As -> As_<group>
isas = tax.label.eq("As")
tax.loc[isas, "label"] = tax.loc[isas, "taxID"].map(lambda t: as_label(lineage(t)))
tax = (pd.concat([tax, pd.DataFrame({
          "taxID": [MGYP4137846_TAXID] + RADIX + [0],
          "label": ["uA", "E_O_Metazoa", "E_O_Metazoa", "U"]})], ignore_index=True)
         .drop_duplicates(subset=["taxID"], keep="first"))
assert tax.taxID.is_unique
tax.to_csv("afesm/taxmap.tsv", sep="\t", header=False, index=False)
print(tax.label.value_counts().to_string())
assert not tax.label.eq("As").any()
assert not tax.label.str.contains(r"\s").any(), "labels must not contain whitespace"

label
E_O_Metazoa               1207648
B                          695662
E_Archaeplastida           276218
V                          246272
E_O_Fungi                  197825
E_Sar_Alveolata             20776
A                           20480
U                           20081
E_Sar_Stramenopiles         14885
E_Sar_Rhizaria               4079
E_Amoebozoa                  3745
E_basal_Discoba              2989
uA                           2107
E_basal_Metamonada           1175
E_Haptista                    919
E_Unassigned                  578
E_O_other                     537
E_Cryptophyceae               528
As_Lokiarchaeia               193
As_Heimdallarchaeia           150
As_Thorarchaeia                90
E_Apusozoa                     60
E_Ancyromonadida               54
As_Sifarchaeia                 38
As_Njordarchaeia               33
E_CRuMs                        31
As_Jordarchaeia                29
As_Hermodarchaeia              27
E_Provora                      20
As_Baldr

## 2b · Reclassification of taxID 4137846 members
AFESM files 1,035,956 proteins under the composite taxID 4137846 ("cellular organisms; Archaea").

More detailed information in `afesm-asgard-eukaryotic-clusters/data/taxon4137846
/README.md`

`categorize_lineage()` labels that taxID itself `uA` (lineage stops at the domain), so without this step every member would be `uA`. Each UniProt member is reclassified from its own record with the same `categorize_lineage()` rule:

| Source | Criterion | Label | Proteins (all AFESM) |
|---|---|---|---|
| UniProtKB | lineage contains Promethearchaeati / Asgard / Panguiarchaeales | As | 18,659 |
| UniProtKB | resolved below the domain level | A | 574,723 |
| UniSave (deleted from UniProtKB) | archived lineage at phylum level or below | A | 50,213 |
| UniProtKB | domain only, "uncultured archaeon" (NCBI 115547) | uA | 95,122 |
| MGnify (MGYP) | no UniProt record (not overridden; 4137846 → uA) | uA | 297,239 |

Asgard members are further split by class from their UniProt lineage (`asgard_class()`). Each member gets a placeholder taxID carrying its label (`afesm/asgard_override.tsv`, `memId → taxID`); the per-cluster pass applies it before classification.

In [4]:
#@title 2b. Reclassify taxID-4137846 members from their own UniProt / UniSave lineage
OVR = "afesm/asgard_override.tsv"
# placeholder taxIDs, one per label, appended to the taxmap
FIX_TAXID = {"A": 41378461, "As": 41378462, "uA": MGYP4137846_TAXID}
EXPECTED  = {"A": 574_723 + 50_213, "uA": 95_122, "As": 18_659}   # all AFESM, before any filter

print("label of taxID 4137846 itself:", tax.set_index("taxID").label.get(4137846, "not in dump"))
if APPLY_ASGARD_FIX:
    src = "afesm/memId_with_gtdbtaxid4137846.Uniprot.tsv.gz"
    if not os.path.exists(src):
        subprocess.run(["gdown", "--quiet", f"https://drive.google.com/uc?id={ASGARD_FIX_GDRIVE_ID}",
                        "-O", src], check=True)
    # columns: From | Entry | Protein names | Taxonomic lineage | Organism | Organism (ID)
    u = pd.read_csv(src, sep="\t", dtype=str, keep_default_na=False)
    deleted = u["Protein names"].eq("deleted")
    # "cellular organisms (no rank), Archaea (domain), ..." -> "cellular organisms; Archaea; ..."
    lin = (u["Taxonomic lineage"].str.replace(r"\s*\([^)]*\)", "", regex=True)
                                 .str.replace(", ", "; ", regex=False))
    # deleted entries carry no lineage here; their UniSave lineages (deleted_memId_with_taxids.tsv)
    # all resolve to phylum level or below with no Asgard keyword -> A
    lab = lin.map(categorize_lineage).where(~deleted, "A")
    got = lab.value_counts().to_dict()
    assert got == EXPECTED, f"unexpected 4137846 reclassification: {got}"
    assert u["From"].is_unique

    # Asgard members: class from their own UniProt lineage -> one placeholder taxID per class
    fine = lab.copy()
    isas = lab.eq("As")
    fine[isas] = lin[isas].map(as_label)
    AS_FIX_TAXID = {l: 413784620 + i for i, l in enumerate(sorted(fine[isas].unique()))}
    ovr = pd.DataFrame({"memId": u["From"],
                        "taxID": fine.map({"A": FIX_TAXID["A"], "uA": FIX_TAXID["uA"], **AS_FIX_TAXID})})
    assert ovr.taxID.notna().all()
    ovr.to_csv(OVR, sep="\t", header=False, index=False)

    new = pd.DataFrame({"taxID": [FIX_TAXID["A"]] + list(AS_FIX_TAXID.values()),
                        "label": ["A"] + list(AS_FIX_TAXID)})
    assert not new.taxID.isin(tax.taxID).any(), "placeholder taxID collides with the dump"
    tax = pd.concat([tax, new], ignore_index=True)
    assert tax.set_index("taxID").label[MGYP4137846_TAXID] == "uA"
    tax.to_csv("afesm/taxmap.tsv", sep="\t", header=False, index=False)

    print(f"{len(ovr):,} members of 4137846 reassigned:")
    print(lab.value_counts().to_string())
    print("Asgard members of 4137846 by class:")
    print(fine[isas].value_counts().to_string())
else:
    open(OVR, "w").close()
    print("APPLY_ASGARD_FIX=False -> all taxID-4137846 members keep the label of 4137846")


label of taxID 4137846 itself: uA
738,717 members of 4137846 reassigned:
Taxonomic lineage
A     624936
uA     95122
As     18659
Asgard members of 4137846 by class:
Taxonomic lineage
As_Heimdallarchaeia    18659


## 3 · Download the member table
7.2 GB. Resumes partial downloads and verifies size + gzip integrity. If it fails, just re-run the cell.

In [5]:
#@title 3. Download the member table  (7.2 GB — resumes and verifies)
dst = "afesm/" + MEMBERS
for attempt in range(1, 7):
    if os.path.exists(dst) and os.path.getsize(dst) == MEMBERS_BYTES:
        break
    # -C - resumes a partial file. The server drops long transfers often enough that this
    # loop is load-bearing: a truncated .gz still decompresses for millions of rows before
    # failing, so a size check is the only reliable guard.
    subprocess.run(["curl", "-s", "-C", "-", "-o", dst, BASE + MEMBERS], check=False)
    print(f"attempt {attempt}: {os.path.getsize(dst):,} / {MEMBERS_BYTES:,} bytes")
assert os.path.getsize(dst) == MEMBERS_BYTES, "download incomplete — re-run this cell"
subprocess.run(["gzip", "-t", dst], check=True)
print("archive verified")

attempt 1: 2,882,666,496 / 7,186,445,769 bytes
attempt 2: 7,186,445,769 / 7,186,445,769 bytes
archive verified


## 4 · Per-cluster pass
Single streaming `awk` pass over 820M rows (~60–90 min). Memory stays constant.

In [6]:
#@title 4. Per-cluster pass  (~60-90 min)
# Rows are grouped by repId in the source file, so one streaming pass suffices and memory
# stays O(1): counters are flushed each time repId changes. `delete c` rather than zeroing
# every key matters here — most of the 157M clusters touch only one or two categories.
# Files are told apart by FILENAME, not NR==FNR, because the override file may be empty.
AWK = r"""
BEGIN {
  n = split(COLS, COL, " ")   # category columns, passed in from Python
  h = "repId\ttotal\ttotal_all\tn_flag3\tcluFlag12\tE_total\tAs"
  for (i = 1; i <= n; i++) h = h "\t" COL[i]
  print h
}
function emit(   i, eu, asg, tot, line, sum) {
  if (cur == "") return
  nblk++
  tot = DROP3 ? tot_all - n3 : tot_all
  if (tot < MINT) { return }
  kept++
  eu = 0
  asg = 0
  for (i = 1; i <= n; i++) {
    if (substr(COL[i], 1, 2) == "E_")  eu  += c[COL[i]]
    if (substr(COL[i], 1, 3) == "As_") asg += c[COL[i]]
  }
  line = cur "\t" tot "\t" tot_all "\t" n3 "\t" (f12 ? 1 : 0) "\t" eu "\t" asg
  sum = 0
  for (i = 1; i <= n; i++) { line = line "\t" (c[COL[i]] + 0); sum += c[COL[i]] }
  if (sum != tot) bad++
  print line
}
FILENAME == TM { m[$1] = $2; next }
FILENAME == OV { ov[$1] = $2; next }
{
  if ($1 != cur) { emit(); cur = $1; tot_all = 0; n3 = 0; f12 = 0; delete c }
  tot_all++
  if ($3 == 3)            n3++
  else if ($3 == 1 || $3 == 2) f12 = 1
  if (DROP3 && $3 == 3) next
  tid = $4
  if ($2 in ov) { tid = ov[$2]; novr++ }   # 4137846 reclassification (step 2b)
  if (tid == "4137846" && $2 ~ /^MGYP/) { c["uA"]++; next }   # MGYP on 4137846 -> unassigned archaea
  lab = (tid in m) ? m[tid] : "NOMAP"
  if (lab == "NOMAP") nomap++; else c[lab]++
}
END {
  emit()
  print "#BLOCKS",    nblk      > "/dev/stderr"
  print "#KEPT",      kept + 0  > "/dev/stderr"
  print "#NOMAP",     nomap + 0 > "/dev/stderr"
  print "#ROWSUM_BAD", bad + 0  > "/dev/stderr"
  print "#OVERRIDDEN", novr + 0 > "/dev/stderr"
}
"""
with open("afesm/cluster.awk", "w") as fh:
    fh.write(AWK)

E_COLS = ("E_O_Metazoa E_O_Fungi E_O_other E_Sar_Alveolata E_Sar_Stramenopiles E_Sar_Rhizaria "
          "E_Sar_unknown E_Archaeplastida E_Amoebozoa E_Haptista E_Cryptophyceae E_Apusozoa "
          "E_basal_Discoba E_basal_Metamonada E_basal_Malawimonadida E_CRuMs E_Ancyromonadida "
          "E_Breviatea E_Provora E_Hemimastigophora E_Unassigned").split()
AS_COLS = sorted(l for l in tax.label.unique() if l.startswith("As_"))
COLS = ["B", "A", "uA", "V", "U"] + AS_COLS + E_COLS
assert set(tax.label) <= set(COLS), set(tax.label) - set(COLS)
assert not any(any(ch.isspace() for ch in c) for c in COLS), "column names must not contain whitespace"
print("Asgard columns:", AS_COLS)
print("streaming 820M rows...")
subprocess.run(["bash", "-lc",
    f"gzip -dc {dst} | awk -F'\\t' -v MINT={MIN_TOTAL} -v DROP3={int(DROP_FLAG3)} "
    f"-v TM=afesm/taxmap.tsv -v OV={OVR} -v COLS='{' '.join(COLS)}' "
    f"-f afesm/cluster.awk afesm/taxmap.tsv {OVR} - 2> afesm/pass_stats.txt | gzip -c > {OUT}"],
    check=True)

stats = {}
for line in open("afesm/pass_stats.txt"):
    if line.startswith("#"):
        key, val = line.split()
        stats[key[1:]] = int(val)
print(stats)
assert stats["BLOCKS"] == N_CLUSTERS, f"saw {stats['BLOCKS']} clusters, expected {N_CLUSTERS}"
assert stats["NOMAP"] == 0,      "unmapped taxIDs — check afesm/taxmap.tsv"
assert stats["ROWSUM_BAD"] == 0, "a row's category counts did not sum to its total"
print(f"{stats['OVERRIDDEN']:,} member rows reassigned by step 2b (all clusters, before MIN_TOTAL; flag-3 rows excluded when DROP_FLAG3)")

Asgard columns: ['As_Asgardarchaeia', 'As_Baldrarchaeia', 'As_Freyarchaeia', 'As_Friggarchaeia', 'As_Gefionarchaeia', 'As_Heimdallarchaeia', 'As_Hermodarchaeia', 'As_Idunnarchaeia', 'As_JAGGXT01', 'As_Jordarchaeia', 'As_Lokiarchaeia', 'As_Njordarchaeia', 'As_Odinarchaeia', 'As_Sifarchaeia', 'As_Thorarchaeia', 'As_Wukongarchaeia', 'As_unresolved']
streaming 820M rows...
{'BLOCKS': 157068338, 'KEPT': 1894714, 'NOMAP': 0, 'ROWSUM_BAD': 0, 'OVERRIDDEN': 691966}
691,966 member rows reassigned by step 2b (all clusters, before MIN_TOTAL; flag-3 rows excluded when DROP_FLAG3)


## 5 · Load and check
Sanity checks and summary counts.

In [7]:
#@title 5. Load and check
cc = pd.read_csv(OUT, sep="\t")
ECOLS = [c for c in cc.columns if c.startswith("E_") and c != "E_total"]   # exclude E_total!
ASCOLS = [c for c in cc.columns if c.startswith("As_")]
CAT   = ["B", "A", "uA", "V", "U"] + ASCOLS + ECOLS

assert len(ECOLS) == 21
assert "E_Other" not in cc.columns, "E_Other should be merged into E_Unassigned"
assert cc.repId.is_unique
assert cc.total.min() >= MIN_TOTAL
assert (cc[CAT].sum(axis=1) == cc.total).all(),    "category columns must sum to total"
assert (cc.E_total == cc[ECOLS].sum(axis=1)).all(), "E_total must equal its 21 components"
assert (cc.As == cc[ASCOLS].sum(axis=1)).all(),     "As must equal the sum of its As_ classes"
if not DROP_FLAG3:
    assert (cc.total == cc.total_all).all()

if MIN_TOTAL == 20:          # known-good reference points
    if DROP_FLAG3:
        assert len(cc) == 1894714, len(cc)
    else:
        assert len(cc) == 2355093, len(cc)
        assert int(cc.cluFlag12.sum()) == 1894714
        # every flag-3 member lives in a flag-3-only cluster
        assert int(cc.loc[cc.cluFlag12 == 0, "total"].sum()) == 32213031

print(f"{len(cc):,} clusters x {cc.shape[1]} columns")
print(f"  cluFlag12 == 1: {int(cc.cluFlag12.sum()):,}"
      "   <- notebook 1's scope; filter on this for the pLDDT>60 subset")
print(f"  proteins covered: {int(cc.total.sum()):,}")
print("\nproteins by category:")
tot = cc[CAT + ["As"]].sum()
for k in ["B", "U", "A", "uA", "As", "V"]:
    print(f"  {k:14s} {int(tot[k]):>12,}")
print(f"  {'E (all)':14s} {int(cc.E_total.sum()):>12,}")
print("\nAsgard proteins by class (clusters with >=1 such protein):")
for k in ASCOLS:
    print(f"  {k:18s} {int(tot[k]):>9,}   {int((cc[k] > 0).sum()):>7,}")

sc = cc[cc.cluFlag12 == 1]
print(f"\nAsgard-containing clusters (As > 0): {int((cc.As > 0).sum()):,}"
      f"   | cluFlag12 == 1: {int((sc.As > 0).sum()):,}")
print(f"clusters with unassigned archaea (uA > 0): {int((cc.uA > 0).sum()):,}"
      f"   | cluFlag12 == 1: {int((sc.uA > 0).sum()):,}")

# Notebook 1 reference (1_clusters_with_20counts_treemap_for_taxonomy_4137846fix): protein totals in the
# cluFlag12 == 1, >=20 scope. Both notebooks apply the same 4137846 fix, so A and uA match separately; the extra
# domain-only uA rule here (environmental samples / unclassified Archaea / incertae sedis) adds 0 proteins in this scope.
# V is not printed in notebook 1; it is derived as 555,961,987 (flag-1 + flag-2 rows) minus the rest.
if MIN_TOTAL == 20:
    REF = {"As": 442_684, "A": 12_369_794, "uA": 307_306,
           "B": 448_066_625, "E": 44_319_915,
           "U": 46_887_887, "V": 3_567_776}
    got = {"As": sc.As.sum(), "A": sc.A.sum(), "uA": sc.uA.sum(), "B": sc.B.sum(), "E": sc.E_total.sum(),
           "U": sc.U.sum(), "V": sc.V.sum()}
    cmp = pd.DataFrame({"notebook_1": REF, "this_run": {k: int(v) for k, v in got.items()}})
    cmp["diff"] = cmp.this_run - cmp.notebook_1
    print("\nvs notebook 1 (cluFlag12 == 1 scope):")
    print(cmp.to_string())
    check = [k for k in REF if APPLY_ASGARD_FIX or k not in ("As", "A", "uA", "U")]
    assert (cmp.loc[check, "diff"] == 0).all(), "does not reproduce notebook 1"

cc.head()

1,894,714 clusters x 50 columns
  cluFlag12 == 1: 1,894,714   <- notebook 1's scope; filter on this for the pLDDT>60 subset
  proteins covered: 555,961,987

proteins by category:
  B               448,066,625
  U                46,887,887
  A                12,369,794
  uA                  307,306
  As                  442,684
  V                 3,567,776
  E (all)          44,319,915

Asgard proteins by class (clusters with >=1 such protein):
  As_Asgardarchaeia          0         0
  As_Baldrarchaeia      13,891     4,741
  As_Freyarchaeia            0         0
  As_Friggarchaeia           0         0
  As_Gefionarchaeia          0         0
  As_Heimdallarchaeia    52,508    17,602
  As_Hermodarchaeia      2,626     1,461
  As_Idunnarchaeia           0         0
  As_JAGGXT01            4,462     2,151
  As_Jordarchaeia       13,863     4,931
  As_Lokiarchaeia      118,024    23,994
  As_Njordarchaeia      38,024     7,926
  As_Odinarchaeia        2,945     1,637
  As_Sifarchaeia 

,repId,total,total_all,n_flag3,cluFlag12,E_total,As,B,A,uA,...,E_Apusozoa,E_basal_Discoba,E_basal_Metamonada,E_basal_Malawimonadida,E_CRuMs,E_Ancyromonadida,E_Breviatea,E_Provora,E_Hemimastigophora,E_Unassigned
0,A0A009EU25,43,43,0,1,0,0,38,0,0,...,0,0,0,0,0,0,0,0,0,0
1,A0A009FAC9,74,74,0,1,0,0,64,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A0A009FFJ9,159,159,0,1,0,0,155,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A0A009FRL1,130,130,0,1,1,0,32,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A0A009GDC3,25,25,0,1,0,0,25,0,0,...,0,0,0,0,0,0,0,0,0,0


## 5b · Asgard classes, collapsed
`AS_GROUP` maps the fine `As_` columns onto the grouping you want to report; columns not listed keep their own name. Only this cell needs re-running when the grouping changes.

In [8]:
#@title 5b. Asgard counts by class (collapsible)
# Columns are already GTDB classes. Use this only to pool further, e.g. Njordarchaeia with
# Heimdallarchaeia, or small classes into one bin. Leave empty to report by class.
AS_GROUP = {
    # "As_Njordarchaeia": "As_Heimdallarchaeia",
    # "As_Freyarchaeia": "...", "As_Friggarchaeia": "...", "As_Gefionarchaeia": "...", "As_Idunnarchaeia": "...",
    # "As_JAGGXT01": "As_other", "As_Baldrarchaeia": "As_other",
}
asg = cc[ASCOLS].T.groupby(lambda c: AS_GROUP.get(c, c)).sum().T
assert (asg.sum(axis=1) == cc.As).all()
summary = pd.DataFrame({
    "proteins": asg.sum(),
    "clusters_with_any": (asg > 0).sum(),
    "clusters_with_E": ((asg > 0) & (cc.E_total.values[:, None] > 0)).sum(),
}).sort_values("proteins", ascending=False)
summary["pct_of_As"] = (100 * summary.proteins / summary.proteins.sum()).round(2)
print(summary.to_string())
print("total As:", int(summary.proteins.sum()))

                     proteins  clusters_with_any  clusters_with_E  pct_of_As
As_Lokiarchaeia        118024              23994            12045      26.66
As_unresolved           94559              18263             9270      21.36
As_Thorarchaeia         83349              12858             6559      18.83
As_Heimdallarchaeia     52508              17602             9515      11.86
As_Njordarchaeia        38024               7926             4300       8.59
As_Sifarchaeia          18103               3734             1638       4.09
As_Baldrarchaeia        13891               4741             2401       3.14
As_Jordarchaeia         13863               4931             2690       3.13
As_JAGGXT01              4462               2151             1220       1.01
As_Odinarchaeia          2945               1637              996       0.67
As_Hermodarchaeia        2626               1461              861       0.59
As_Wukongarchaeia         330                221              130       0.07

## 6 · Slim table
Drops `total_all`, `n_flag3`, `cluFlag12` (constant for `MIN_TOTAL=20, DROP_FLAG3=True`) and every all-zero column; writes `….slim.tsv.gz`.

In [27]:
#@title 6. Slim table: drop redundant and all-zero columns
assert (cc.total == cc.total_all).all() and (cc.n_flag3 == 0).all() and (cc.cluFlag12 == 1).all(), \
    "total_all / n_flag3 / cluFlag12 are not redundant in this run - keep them"

num = cc.columns.drop("repId")
zero = [c for c in num if (cc[c] == 0).all()]
slim = cc.drop(columns=["total_all", "n_flag3", "cluFlag12"] + zero)

# sums still hold after dropping
AS = [c for c in slim.columns if c.startswith("As_")]
E  = [c for c in slim.columns if c.startswith("E_") and c != "E_total"]
assert (slim[["B", "A", "uA", "V", "U"] + AS + E].sum(axis=1) == slim.total).all()
assert (slim[AS].sum(axis=1) == slim.As).all() and (slim[E].sum(axis=1) == slim.E_total).all()

OUT_SLIM = OUT.replace(".tsv.gz", ".slim.tsv.gz")
slim.to_csv(OUT_SLIM, sep="\t", index=False, compression="gzip")
print("dropped all-zero:", zero)
print(f"{OUT_SLIM}: {len(slim):,} rows x {slim.shape[1]} columns")

dropped all-zero: ['n_flag3', 'As_Asgardarchaeia', 'As_Freyarchaeia', 'As_Friggarchaeia', 'As_Gefionarchaeia', 'As_Idunnarchaeia', 'E_Hemimastigophora']
afesm_cluster_composition.geq20.noFlag3.slim.tsv.gz: 1,894,714 rows x 41 columns


## Exploration
Optional checks behind the classification choices (unresolved archaea, NAG2, eukaryotes without a supergroup, Thermoproteota phylum-only proteins near Panguiarchaeales). Not needed to build the table.

In [9]:
ua = tax[tax.label == "uA"].copy()
ua["lin"] = ua.taxID.map(lineage)
parts = ua.lin.str.split("; ")
ua["rank2"] = parts.str[2]          # rank after "cellular organisms; Archaea"
ua["rank3"] = parts.str[3]
print(ua.rank2.value_counts(dropna=False))
print(ua[ua.rank2.isin(["unclassified Archaea", "Archaea incertae sedis"])]
        .rank3.value_counts().head(30))

rank2
environmental samples     1733
unclassified Archaea       361
Archaea incertae sedis      10
NaN                          3
Name: count, dtype: int64
rank3
Archaea candidate phyla     9
archaeon DSFBPS_UR1G        1
archaeon DSFBPS_UR1E        1
archaeon DSFBPS_UR1D        1
archaeon DSFBPS_UR1C        1
archaeon DSFBPS_UR1B        1
archaeon DSFBPS_UR1A        1
archaeon DSFBPS_4R3A        1
archaeon DSFBPG_URC2C       1
archaeon DSFBPG_UR3F        1
archaeon DSFBPG_UR3E        1
archaeon DSFBPG_UR3D        1
archaeon DSFBPG_UR3C        1
archaeon DSFBPG_UR3B        1
archaeon DSFBPG_UR3A        1
archaeon DSFBPG_UR2G        1
archaeon DSFBPG_UR2F        1
archaeon DSFBPG_UR2E        1
archaeon DSFBPG_UR2D        1
archaeon DSFBPG_UR2B        1
archaeon DSFBPG_UR2A        1
archaeon M21                1
archaeon DSFBPS_UR1F        1
archaeon SCGC AC-708-N22    1
archaeon SCGC AC-708-M03    1
archaeon SCGC AC-708-L17    1
archaeon SCGC AC-335-M03    1
archaeon SCGC AC-335-L21    

In [10]:
cp = ua[ua.rank3 == "Archaea candidate phyla"]
print(cp[["taxID", "lin"]].to_string())

named = ua[(ua.rank2 == "unclassified Archaea") & ~ua.rank3.str.startswith("archaeon", na=False)]
print(named.rank3.value_counts().to_string())

           taxID                                                                                                                                                             lin
1157019  1448937                                                             cellular organisms; Archaea; Archaea incertae sedis; Archaea candidate phyla; candidate phylum NAG2
1157157  1449106       cellular organisms; Archaea; Archaea incertae sedis; Archaea candidate phyla; candidate phylum NAG2; candidate phylum NAG2 archaeon FF85/JGI 000156CP-C16
1157158  1449107  cellular organisms; Archaea; Archaea incertae sedis; Archaea candidate phyla; candidate phylum NAG2; candidate phylum NAG2 archaeon MK4_SK304/JGI 000156CP-M15
1157159  1449108  cellular organisms; Archaea; Archaea incertae sedis; Archaea candidate phyla; candidate phylum NAG2; candidate phylum NAG2 archaeon MK4_SK304/JGI 000156CP-M16
1157160  1449109     cellular organisms; Archaea; Archaea incertae sedis; Archaea candidate phyla; candidate phylum

In [11]:
!zcat afesm/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz | awk -F'\t' '$4==1448937 || ($4>=1449106 && $4<=1449112){n[$4]++; t++} END{for(k in n) print k, n[k]; print "total NAG2 proteins:", t+0}'

total NAG2 proteins: 0


In [12]:
# what is pooled in E_Unassigned: rank right after Eukaryota (NaN = domain-only taxon 2759)
eu = tax[tax.label == "E_Unassigned"].copy()
eu["lin"] = eu.taxID.map(lineage)
eu["rank2"] = eu.lin.str.split("; ").str[2]
print(eu.rank2.value_counts(dropna=False).to_string())

print("E_Unassigned proteins (filtered table):", int(cc.E_Unassigned.sum()))

rank2
environmental samples       298
unclassified eukaryotes     275
Eukaryota incertae sedis      4
NaN                           1
E_Unassigned proteins (filtered table): 494822


In [13]:
# proteins in E_Unassigned that sit on the domain-only taxon 2759 (formerly the whole of E_Unassigned)
print(eu.loc[eu.rank2.isna(), "taxID"].tolist())

[2759]


In [14]:
print(len(eu), "taxa pooled in E_Unassigned; examples:")
for t in eu.taxID.head(10): print(t, lineage(t))

578 taxa pooled in E_Unassigned; examples:
2759 cellular organisms; Eukaryota
42452 cellular organisms; Eukaryota; unclassified eukaryotes
42453 cellular organisms; Eukaryota; environmental samples; named eukaryotic clones; unidentified eukaryote clone OM13
42454 cellular organisms; Eukaryota; environmental samples; named eukaryotic clones; unidentified eukaryote clone OM19
42455 cellular organisms; Eukaryota; environmental samples; named eukaryotic clones; unidentified eukaryote clone OM22
42456 cellular organisms; Eukaryota; environmental samples; named eukaryotic clones; unidentified eukaryote clone OM20
42457 cellular organisms; Eukaryota; environmental samples; named eukaryotic clones; unidentified eukaryote clone OM21
56059 cellular organisms; Eukaryota; unclassified eukaryotes; unidentified protist 56059
61665 cellular organisms; Eukaryota; environmental samples; named eukaryotic clones; environmental clone OCS162
61666 cellular organisms; Eukaryota; environmental samples; named

In [15]:
!zcat afesm/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz | awk -F'\t' -v T=2759 '$4==T && $3!=3 {if ($2 ~ /^MGYP/) m++; else {u++; print $2 > "afesm/euk_domain_uniprot.txt"}} END{print "MGYP:", m+0, " UniProt:", u+0}'

MGYP: 774499  UniProt: 0


`E_Other` (environmental samples / unclassified eukaryotes / incertae sedis) is now merged into `E_Unassigned` ("unresolved eukaryotes") at classification time — see cell 2.

In [16]:
def path(t):
    """taxids root -> tip"""
    t = merged.get(t, t); out = []; seen = set()
    while t in nodes and t not in seen and t != 1:
        seen.add(t); out.append(t); t = nodes[t]
    return out[::-1]

# every ancestor of an Asgard-labelled taxon (e.g. GTDB Thermoproteota, JANJXX01 above Panguiarchaeales)
anc = set()
for t in tax.loc[tax.label == "As", "taxID"]:
    anc.update(path(t)[:-1])

# deepest named rank of a taxon, ignoring placeholder ranks below it
PLACE = ("unclassified", "environmental samples", "incertae sedis", "uncultured")
def last_resolved(t):
    last = None
    for x in path(t):
        if any(k in names.get(x, "").lower() for k in PLACE): break
        last = x
    return last

amb = [t for t in tax.loc[tax.label == "A", "taxID"] if last_resolved(t) in anc]
pd.DataFrame({"taxID": amb, "lin": [lineage(t) for t in amb]}) \
  .to_csv("afesm/A_ambiguous_taxids.tsv", sep="\t", header=False, index=False)
print(len(amb), "A taxids that could hide Asgard")
for t in amb[:20]: print(t, lineage(t))

0 A taxids that could hide Asgard


In [17]:
cc.repId.to_csv("afesm/keep_repIds.txt", index=False, header=False)

In [18]:
!zcat afesm/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz | awk -F'\t' 'FILENAME==ARGV[1]{k[$1];next} FILENAME==ARGV[2]{s[$1]=$2;next} $3!=3 && ($1 in k) && ($4 in s){n[$4]++} END{for(t in n) print n[t]"\t"t"\t"s[t]}' afesm/keep_repIds.txt afesm/A_ambiguous_taxids.tsv - | sort -k1,1nr > afesm/A_ambiguous_counts.tsv
!awk -F'\t' '{s+=$1} END{print "proteins in A that could be Panguiarchaeales:", s+0}' afesm/A_ambiguous_counts.tsv
!cat afesm/A_ambiguous_counts.tsv

proteins in A that could be Panguiarchaeales: 0


In [19]:
kids = [t for t, p in nodes.items() if p == 4138200]
for t in kids: print(t, names.get(t), "->", tax.set_index("taxID").label.get(t))

4138201 Panguiarchaeales -> As_Njordarchaeia
4146198 JAMOTR01 -> A


In [20]:
# taxa below GTDB Thermoproteota, with their label (As = Panguiarchaeales side)
desc = tax[[4137859 in path(t)[:-1] for t in tax.taxID]][["taxID", "label"]]
desc.to_csv("afesm/thermo_desc.tsv", sep="\t", header=False, index=False)
cc[["repId", "As"]].to_csv("afesm/keep_repIds_As.tsv", sep="\t", header=False, index=False)
print(desc.label.value_counts())

label
A                   1765
As_Njordarchaeia      18
Name: count, dtype: int64


In [21]:
!zcat afesm/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz | awk -F'\t' 'FILENAME==ARGV[1]{k[$1]=$2;next} FILENAME==ARGV[2]{d[$1]=$2;next} $3!=3 && ($1 in k){ if($4 in d) lab[d[$4]]++; if($4=="4137859"){ph++; if(k[$1]>0) phAs++} } END{s=lab["As"]/(lab["As"]+lab["A"]); printf "Thermoproteota resolved below phylum: As=%d A=%d  (Pangui share %.4f)\nphylum-only proteins: %d  -> expected Panguiarchaeales ~ %.0f\nphylum-only proteins in Asgard-containing clusters: %d\n", lab["As"], lab["A"], s, ph, ph*s, phAs}' afesm/keep_repIds_As.tsv afesm/thermo_desc.tsv -

Thermoproteota resolved below phylum: As=0 A=2861594  (Pangui share 0.0000)
phylum-only proteins: 214134  -> expected Panguiarchaeales ~ 0
phylum-only proteins in Asgard-containing clusters: 161083


In [22]:
!zcat afesm/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz | awk -F'\t' 'FILENAME==ARGV[1]{k[$1];next} $3!=3 && ($1 in k) && $4=="4137859"{n[$1]++} END{for(r in n) print r"\t"n[r]}' afesm/keep_repIds.txt - > afesm/thermo_phylum_per_cluster.tsv

In [23]:
tp = pd.read_csv("afesm/thermo_phylum_per_cluster.tsv", sep="\t", names=["repId", "thermo_ph"])
d = cc.merge(tp, on="repId", how="left").fillna({"thermo_ph": 0})
assert d.thermo_ph.sum() == 214134

# every A protein in the cluster is a phylum-only Thermoproteota protein
only_tp = (d.As > 0) & (d.B == 0) & (d.A > 0) & (d.A == d.thermo_ph)
other0  = (d.uA == 0) & (d.V == 0) & (d.U == 0)        # stricter: nothing else at all

cases = {
    "As + Thermo-only (E=0)":       only_tp & (d.E_total == 0),
    "E + As + Thermo-only (E>0)":   only_tp & (d.E_total > 0),
}
for name, m in cases.items():
    for label, mm in [("any uA/V/U", m), ("uA=V=U=0", m & other0)]:
        print(f"{name:28s} [{label:10s}] clusters: {int(mm.sum()):>6,}   Thermo-only proteins: {int(d.loc[mm, 'thermo_ph'].sum()):>7,}")

As + Thermo-only (E=0)       [any uA/V/U] clusters:      5   Thermo-only proteins:       7
As + Thermo-only (E=0)       [uA=V=U=0  ] clusters:      0   Thermo-only proteins:       0
E + As + Thermo-only (E>0)   [any uA/V/U] clusters:      1   Thermo-only proteins:       1
E + As + Thermo-only (E>0)   [uA=V=U=0  ] clusters:      1   Thermo-only proteins:       1


In [24]:
m = (d.As > 0) & (d.B == 0) & (d.A > 0) & (d.A == d.thermo_ph) & (d.E_total > 0)
print(d.loc[m].T.to_string())

                        495279
repId                   M7CCK3
total                       47
total_all                   47
n_flag3                      0
cluFlag12                    1
E_total                     43
As                           3
B                            0
A                            1
uA                           0
V                            0
U                            0
As_Asgardarchaeia            0
As_Baldrarchaeia             0
As_Freyarchaeia              0
As_Friggarchaeia             0
As_Gefionarchaeia            0
As_Heimdallarchaeia          0
As_Hermodarchaeia            0
As_Idunnarchaeia             0
As_JAGGXT01                  0
As_Jordarchaeia              1
As_Lokiarchaeia              1
As_Njordarchaeia             0
As_Odinarchaeia              0
As_Sifarchaeia               0
As_Thorarchaeia              1
As_Wukongarchaeia            0
As_unresolved                0
E_O_Metazoa                 17
E_O_Fungi                    7
E_O_othe

In [25]:
%%bash
zcat afesm/3-repId_memId_cluFlag_taxId_biomeId.tsv.gz | awk -F'\t' '$1=="M7CCK3"' > afesm/M7CCK3_members.tsv
echo "== Thermoproteota phylum-only member =="
awk -F'\t' '$4=="4137859"' afesm/M7CCK3_members.tsv
echo "== taxids in the cluster =="
cut -f4 afesm/M7CCK3_members.tsv | sort | uniq -c | sort -nr

== Thermoproteota phylum-only member ==
M7CCK3	A0A7J3S0W9	1	4137859	0
== taxids in the cluster ==
      5 5722
      5 1144522
      4 5963
      2 478820
      2 34880
      2 1754190
      2 1555241
      1 9986
      1 9704
      1 8469
      1 78915
      1 75913
      1 73868
      1 6573
      1 6248
      1 578462
      1 542832
      1 461836
      1 4142034
      1 4139428
      1 4138537
      1 4137859
      1 412133
      1 39432
      1 34884
      1 34506
      1 30611
      1 28743
      1 2715852
      1 174720
      1 13616
      1 111125


In [26]:
!zcat afesm_cluster_composition.geq20.noFlag3.tsv.gz | head -1 | tr '\t' '\n' | cat -n
!zcat afesm_cluster_composition.geq20.noFlag3.tsv.gz | awk -F'\t' '{print NF}' | sort | uniq -c

     1	repId
     2	total
     3	total_all
     4	n_flag3
     5	cluFlag12
     6	E_total
     7	As
     8	B
     9	A
    10	uA
    11	V
    12	U
    13	As_Asgardarchaeia
    14	As_Baldrarchaeia
    15	As_Freyarchaeia
    16	As_Friggarchaeia
    17	As_Gefionarchaeia
    18	As_Heimdallarchaeia
    19	As_Hermodarchaeia
    20	As_Idunnarchaeia
    21	As_JAGGXT01
    22	As_Jordarchaeia
    23	As_Lokiarchaeia
    24	As_Njordarchaeia
    25	As_Odinarchaeia
    26	As_Sifarchaeia
    27	As_Thorarchaeia
    28	As_Wukongarchaeia
    29	As_unresolved
    30	E_O_Metazoa
    31	E_O_Fungi
    32	E_O_other
    33	E_Sar_Alveolata
    34	E_Sar_Stramenopiles
    35	E_Sar_Rhizaria
    36	E_Sar_unknown
    37	E_Archaeplastida
    38	E_Amoebozoa
    39	E_Haptista
    40	E_Cryptophyceae
    41	E_Apusozoa
    42	E_basal_Discoba
    43	E_basal_Metamonada
    44	E_basal_Malawimonadida
    45	E_CRuMs
    46	E_Ancyromonadida
    47	E_Breviatea
    48	E_Provora
    49	E_Hemimastigophora
    50	E_Unassigned
189471